# Batch-run TALOS-N over every test-set entry

Uses the `TalosN` wrapper of makeshift.

## Setup

In [ ]:
import csv
import sys
from pathlib import Path

MAKESHIFT_ROOT = "../../../external/makeshift"
if MAKESHIFT_ROOT not in sys.path:
    sys.path.insert(0, MAKESHIFT_ROOT)

import makeshift as ms  # noqa: F401
from makeshift import talosn
from makeshift.entry import NMRStarEntry

## Paths and config

In [ ]:
ROOT = Path("../../../data/analysis/Fig4/Talos-N_results")  # inputs/, output_new/, talosn_data/ live here
INPUT_DIR = ROOT / "inputs"
OUTPUT_DIR = ROOT / "output_new"
DATA_DIR = ROOT / "talosn_data"
LOG_PATH = ROOT / "run_talosn_batch_log.csv"

FORCE_RERUN = False

## Helpers

In [ ]:
def has_valid_s2(entry_output_dir):
    s2_path = entry_output_dir / "predS2.tab"
    if not s2_path.is_file():
        return False
    lines = s2_path.read_text(encoding="utf-8", errors="ignore").splitlines()
    return any(
        line.strip() and not line.startswith(("REMARK", "DATA", "VARS", "FORMAT"))
        for line in lines
    )


def find_input_str(entry_dir):
    str_files = sorted(entry_dir.glob("*.str"))
    if not str_files:
        raise FileNotFoundError(f"no .str file in {entry_dir}")
    if len(str_files) > 1:
        raise ValueError(f"multiple .str files in {entry_dir}: {str_files}")
    return str_files[0]

## TALOS-N data files

In [ ]:
talosn.engine.ensure_talosn_data(data_dir=DATA_DIR, auto_install=True)

## List entries

In [ ]:
entries = sorted(p.name for p in INPUT_DIR.iterdir() if p.is_dir())
print(f"{len(entries)} entries found in {INPUT_DIR}")

## Run TALOS-N over every entry

In [ ]:
rows = []
for i, entry_id in enumerate(entries, 1):
    entry_input_dir = INPUT_DIR / entry_id
    entry_output_dir = OUTPUT_DIR / entry_id

    if not FORCE_RERUN and has_valid_s2(entry_output_dir):
        print(f"[{i}/{len(entries)}] {entry_id}: skip (predS2.tab already present)")
        rows.append({"entry_id": entry_id, "status": "skipped", "error": ""})
        continue

    try:
        str_path = find_input_str(entry_input_dir)
        star_entry = NMRStarEntry.from_file(str_path, entry_id=entry_id)
        tn = talosn.TalosN.from_entry(star_entry, data_dir=DATA_DIR)
        tn.run(output_dir=entry_output_dir, auto_install=True)

        has_s2 = tn.order_parameters is not None and len(tn.order_parameters) > 0
        status = "ok" if has_s2 else "ran_no_s2"
        print(f"[{i}/{len(entries)}] {entry_id}: {status}")
        rows.append({"entry_id": entry_id, "status": status, "error": ""})

    except Exception as e:
        print(f"[{i}/{len(entries)}] {entry_id}: FAILED - {e}")
        rows.append({"entry_id": entry_id, "status": "failed", "error": str(e)})

## Write log and summary

In [ ]:
with open(LOG_PATH, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["entry_id", "status", "error"])
    w.writeheader()
    w.writerows(rows)

n_ok = sum(r["status"] == "ok" for r in rows)
n_no_s2 = sum(r["status"] == "ran_no_s2" for r in rows)
n_failed = sum(r["status"] == "failed" for r in rows)
n_skipped = sum(r["status"] == "skipped" for r in rows)
print(
    f"\ndone: {n_ok} ok, {n_no_s2} ran but produced no predS2.tab, "
    f"{n_failed} failed to run, {n_skipped} skipped (already had S2)"
)
print(f"log written to {LOG_PATH}")